In [1]:
from os.path import join
import os
from tqdm import tqdm
import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', 10)
from IPython.display import HTML, display
css_style = """
<style>
.scrollable {
    max-height: 1000px; 
    overflow-y: auto;
    position: relative;
}

/* Make the table header sticky */
.scrollable thead th {
    position: sticky;
    top: 0;
    background: rgba(100, 100, 100, 100); 
    z-index: 1;
}


::-webkit-scrollbar {
    -webkit-appearance: none;
}

::-webkit-scrollbar:vertical {
    width: 12px;
}

::-webkit-scrollbar-thumb {
    background-color: rgba(0, 0, 0, .5);
    border-radius: 10px;
    border: 2px solid #ffffff;
}
</style>
"""

def show(df):
    html = css_style + df.to_html(border=0)
    display(HTML('<div class="scrollable">' + html + '</div>'))



In [23]:
df = pd.read_csv('../verosdata.csv')
df

def record_id(patient_id):
    return int(patient_id.split('-')[1])
df['record_id'] = df['record_id'].apply(record_id)

def create_tb_label(df):
    df['confirmed_tb_v2'] = df['confirmed_tb_v2'].astype(int)
    tb_labels = []
    for _, row in df.iterrows():
        tb = row['confirmed_tb_v2']
        if tb:
            tb_labels.append(1)
        else:
            tb_labels.append(0)
    df['TB Label'] = tb_labels
    
    return df


df = create_tb_label(df)


lus_interpretations = []

lus_patterns = {
0: 'Normal A line Pattern',
1: "B-lines",
2: "B-lines",
3: "Small consolidations andor subpleural nodules (< 1cm in height)",
4: "Consolidation of ≥ 1 cm in height",
5: "Pattern A' (pneumothorax)",
6: "Pleural effusion"
}


final_lus_interpretations = [ 'lus_qasd_1_final',
 'lus_qasd_2_final',
 'lus_qaid_1_final',
 'lus_qsld_2_final',
 'lus_qld_1_final',
 'lus_qasg_1_final',
 'lus_qasg_2_final',
 'lus_qaig_1_final',
 'lus_qslg_2_final',
 'lus_qlg_1_final',
 'lus_qpsd_1_final',
 'lus_qpid_1_final',
 'lus_qpsg_1_final',
 'lus_qpig_1_final']


interptosite = {'lus_qasd_1_final': 'APXD',
                'lus_qasd_2_final': 'QASD',
                 'lus_qaid_1_final': 'QAID',
                  'lus_qsld_2_final': 'QSLD',
                  'lus_qld_1_final': 'QLID',
                    'lus_qasg_1_final': 'APXG',
                  'lus_qasg_2_final': 'QASG',
                  'lus_qaig_1_final': 'QAIG',
                  'lus_qslg_2_final': 'QSLG',
                  'lus_qlg_1_final': 'QLIG',
                  'lus_qpsd_1_final': 'QPSD',
                  'lus_qpid_1_final': 'QPID',
                  'lus_qpsg_1_final': 'QPSG',
                  'lus_qpig_1_final': 'QPIG'
                  }

new_labels = ['record_id', 'TB Label'] + final_lus_interpretations #+ final_fash_interpretations
labels = df[new_labels]
#keys = interptosite.keys().tolist()
for col in labels:
    try: 
        newname = interptosite[col]
        labels[newname] = labels[col].fillna(-1)
    except:
        continue
labels = labels.drop(columns=interptosite.keys())
for col in labels.columns:
    labels[col] = labels[col].astype(int)
labels

/var/folders/1c/wzww7zys46v2sv8gcs948xkc0000gn/T/ipykernel_24404/1186749727.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labels[newname] = labels[col].fillna(-1)
/var/folders/1c/wzww7zys46v2sv8gcs948xkc0000gn/T/ipykernel_24404/1186749727.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labels[newname] = labels[col].fillna(-1)
/var/folders/1c/wzww7zys46v2sv8gcs948xkc0000gn/T/ipykernel_24404/1186749727.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

,record_id,TB Label,APXD,QASD,QAID,QSLD,QLID,APXG,QASG,QAIG,QSLG,QLIG,QPSD,QPID,QPSG,QPIG
0,1,0,0,0,1,-1,1,2,3,1,-1,1,0,1,0,1
1,4,0,0,0,0,-1,0,0,0,2,-1,2,0,0,0,0
2,5,1,0,0,0,-1,0,0,1,3,-1,1,0,0,0,3
3,7,1,0,0,0,-1,1,0,2,0,-1,4,0,1,0,0
4,9,0,0,0,0,-1,0,0,0,0,-1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,759,0,1,3,3,3,0,0,3,3,0,1,0,2,4,0
500,760,1,0,0,1,0,1,0,0,0,0,1,1,0,0,0
501,761,1,4,1,2,4,1,3,4,0,3,4,0,0,0,0
502,762,1,2,3,2,4,3,1,1,1,1,6,4,3,2,2


In [24]:
sites = ['APXD', 'QASD', 'QAID', 'QSLD', 'QLID',
         'APXG', 'QASG', 'QAIG', 'QSLG', 'QLIG', 
         'QPSD', 'QPID', 'QPSG', 'QPIG']

# Define the get_number_value function
def get_number_value(numb):
    lus_patterns = {
        int(-1): 'Not measured',
        int(0): 'A-line',
        int(1): "B-lines",
        int(2): "B-lines",
        int(3): "Consolidations or Nodules",
        int(4):  "Consolidations or Nodules",
        int(5): "Pattern A' (pneumothorax)",
        int(6): "Pleural effusion"
    }
    return lus_patterns.get(numb, 'Unknown')

# Apply get_number_value to each site column
for site in sites:
    if site in labels.columns:
        labels[site] = labels[site].astype(int)
        labels[site] = labels[site].apply(get_number_value)

# Convert categorical variables to dummy/indicator variables
for site in sites:
    if site in labels.columns:
        newdf = pd.get_dummies(labels[site], prefix=site)
        labels = pd.concat([labels, newdf], axis=1)
        labels = labels.drop(columns=[site])
for col in labels.columns.tolist():
    labels[col] = labels[col].astype(int)
# Print the updated labels DataFrame to check the result
show(labels)

,record_id,TB Label,APXD_A-line,APXD_B-lines,APXD_Consolidations or Nodules,APXD_Pleural effusion,QASD_A-line,QASD_B-lines,QASD_Consolidations or Nodules,QASD_Pleural effusion,QAID_A-line,QAID_B-lines,QAID_Consolidations or Nodules,QAID_Pleural effusion,QSLD_A-line,QSLD_B-lines,QSLD_Consolidations or Nodules,QSLD_Not measured,QSLD_Pleural effusion,QLID_A-line,QLID_B-lines,QLID_Consolidations or Nodules,QLID_Not measured,QLID_Pleural effusion,APXG_A-line,APXG_B-lines,APXG_Consolidations or Nodules,APXG_Pattern A' (pneumothorax),QASG_A-line,QASG_B-lines,QASG_Consolidations or Nodules,QASG_Pleural effusion,QAIG_A-line,QAIG_B-lines,QAIG_Consolidations or Nodules,QAIG_Not measured,QAIG_Pleural effusion,QSLG_A-line,QSLG_B-lines,QSLG_Consolidations or Nodules,QSLG_Not measured,QSLG_Pleural effusion,QLIG_A-line,QLIG_B-lines,QLIG_Consolidations or Nodules,QLIG_Pleural effusion,QPSD_A-line,QPSD_B-lines,QPSD_Consolidations or Nodules,QPSD_Not measured,QPSD_Pattern A' (pneumothorax),QPSD_Pleural effusion,QPID_A-line,QPID_B-lines,QPID_Consolidations or Nodules,QPID_Not measured,QPID_Pleural effusion,QPSG_A-line,QPSG_B-lines,QPSG_Consolidations or Nodules,QPSG_Not measured,QPSG_Pattern A' (pneumothorax),QPSG_Pleural effusion,QPIG_A-line,QPIG_B-lines,QPIG_Consolidations or Nodules,QPIG_Not measured,QPIG_Pleural effusion
0,1,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0
1,4,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
2,5,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
3,7,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0
4,9,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
5,12,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0
6,14,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
7,20,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
8,23,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
9,24,1,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0


In [9]:

sites = ['APXD', 'QASD', 'QAID', 'QSLD', 'QLID',
         'APXG', 'QASG', 'QAIG', 'QSLG', 'QLIG', 
         'QPSD', 'QPID', 'QPSG', 'QPIG']

# Define the get_number_value function
def get_number_value(numb):
    lus_patterns = {
        -1: 'Not measured',
        0: 'Normal A line Pattern',
        1: "B-lines",
        2: "B-lines",
        3: "Consolidations or Nodules",
        4: "Consolidations or Nodules",
        5: "Pattern A' (pneumothorax)",
        6: "Pleural effusion"
    }
    return lus_patterns.get(numb, 'Unknown')

for site in sites:
    if site in labels.columns:
        labels[site] = labels[site].apply(get_number_value)

# Convert categorical variables to dummy/indicator variables
for site in sites:
    if site in labels.columns:
        newdf = pd.get_dummies(labels[site], prefix=site)
        labels = pd.concat([labels, newdf], axis=1)
        labels = labels.drop(columns=[site])

labels

# for site in sites:
#     labels[site] = labels[site].apply(get_number_value)

# for site in sites:
#     newdf = pd.get_dummies(labels[site], drop_first=True, prefix=site )
#     labels = pd.concat([labels, newdf], axis = 1)
#     labels = labels.drop(columns = [site])

    

#     lus_patterns = {
# -1: 'Not measured',
# 0: 'Normal A line Pattern',
# 1: "Pattern with ≥ 3 B lines per field",
# 2: "Pattern with coalescing B lines ",
# 3: "Pattern with small consolidations and/or subpleural nodules (< 1cm in height)",
# 4: "Consolidation of ≥ 1 cm in height",
# 5: "Pattern A' (pneumothorax)",
# 6: "Pleural effusion"
# }

# def fash_p_effusion(val):
#     if val == 0:
#         return 'Non'
#     if val == 1:
#         return "Yes < 5mm"
#     if val == 2:
#         return "Yes >= 5mm"
#     if val == 3:
#         return "Image Not interpretable"
    
# def fash_others(val):
#     if val == 0:
#         return 'Non'
#     if val == 1:
#         return "Yes"
#     if val == 2:
#         return "Image Not interpretable"




# labels['fash_pericard_effusion_final'] = labels['fash_pericard_effusion_final'].apply(fash_p_effusion)

# fash_cols = ['fash_adp_final', 'fash_rpleural_effusion_final', 'fash_liver_final',
#        'fash_morison_ascites_final', 'fash_lpleural_effusion_final',
#        'fash_spleen_final', 'fash_splenoren_ascites_final',
#        'fash_douglas_ascites_final']
#total = sites #+ fash_cols
#total.extend('fash_pericard_effusion_final')

# for col in fash_cols:
#     labels[col] = labels[col].apply(fash_others)

# for site in sites:
#     newdf = pd.get_dummies(labels[site], drop_first=True, prefix=site )
#     labels = pd.concat([labels, newdf], axis = 1)
#     labels = labels.drop(columns = [site])


,record_id,TB Label,QLID_Consolidation of ≥ 1 cm in height,QLID_Normal A line Pattern,QLID_Not measured,QLID_Pleural effusion,QLID_Small consolidations and/or subpleural nodules (< 1cm in height),APXG_Consolidation of ≥ 1 cm in height,APXG_Normal A line Pattern,APXG_Pattern A' (pneumothorax),APXG_Small consolidations and/or subpleural nodules (< 1cm in height),QASG_Consolidation of ≥ 1 cm in height,QASG_Normal A line Pattern,QASG_Pleural effusion,QASG_Small consolidations and/or subpleural nodules (< 1cm in height),QAIG_Consolidation of ≥ 1 cm in height,QAIG_Normal A line Pattern,QAIG_Not measured,QAIG_Pleural effusion,QAIG_Small consolidations and/or subpleural nodules (< 1cm in height),QSLG_Consolidation of ≥ 1 cm in height,QSLG_Normal A line Pattern,QSLG_Not measured,QSLG_Pleural effusion,QSLG_Small consolidations and/or subpleural nodules (< 1cm in height),QLIG_Consolidation of ≥ 1 cm in height,QLIG_Normal A line Pattern,QLIG_Pleural effusion,QLIG_Small consolidations and/or subpleural nodules (< 1cm in height),QPSD_Consolidation of ≥ 1 cm in height,QPSD_Normal A line Pattern,QPSD_Not measured,QPSD_Pattern A' (pneumothorax),QPSD_Pleural effusion,QPSD_Small consolidations and/or subpleural nodules (< 1cm in height),QPID_Consolidation of ≥ 1 cm in height,QPID_Normal A line Pattern,QPID_Not measured,QPID_Pleural effusion,QPID_Small consolidations and/or subpleural nodules (< 1cm in height),QPSG_Consolidation of ≥ 1 cm in height,QPSG_Normal A line Pattern,QPSG_Not measured,QPSG_Pattern A' (pneumothorax),QPSG_Pleural effusion,QPSG_Small consolidations and/or subpleural nodules (< 1cm in height),QPIG_Consolidation of ≥ 1 cm in height,QPIG_Normal A line Pattern,QPIG_Not measured,QPIG_Pleural effusion,QPIG_Small consolidations and/or subpleural nodules (< 1cm in height)
0,1,0,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
1,4,0,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False
2,5,1,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,True
3,7,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False
4,9,0,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,759,0,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False
500,760,1,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,Fal

In [75]:
# fash_cols.append('fash_pericard_effusion_final')
# for c in fash_cols:
#     newdf = pd.get_dummies(labels[c], drop_first=True, prefix=c )
#     labels = pd.concat([labels, newdf], axis = 1)
#     labels = labels.drop(columns = [c])



labels

In [25]:
FOLDER_DIR = '../CLUSSTER-Benin/images'

files = [f for f in os.listdir(FOLDER_DIR)] 

# Get all dataset names
datasets = sorted({f.split('_')[0] for f in files})
datasets = [int(d) for d in datasets]
print("Number of datasets", len(datasets))
labels = labels[labels['record_id'].isin(datasets)]
for col in labels.columns:
    print(col, labels[col].isna().sum())
labels.to_csv('../CLUSSTER-Benin/labels/newlabelsoh1.csv')

Number of datasets 447
record_id 0
TB Label 0
APXD_A-line 0
APXD_B-lines 0
APXD_Consolidations or Nodules 0
APXD_Pleural effusion 0
QASD_A-line 0
QASD_B-lines 0
QASD_Consolidations or Nodules 0
QASD_Pleural effusion 0
QAID_A-line 0
QAID_B-lines 0
QAID_Consolidations or Nodules 0
QAID_Pleural effusion 0
QSLD_A-line 0
QSLD_B-lines 0
QSLD_Consolidations or Nodules 0
QSLD_Not measured 0
QSLD_Pleural effusion 0
QLID_A-line 0
QLID_B-lines 0
QLID_Consolidations or Nodules 0
QLID_Not measured 0
QLID_Pleural effusion 0
APXG_A-line 0
APXG_B-lines 0
APXG_Consolidations or Nodules 0
APXG_Pattern A' (pneumothorax) 0
QASG_A-line 0
QASG_B-lines 0
QASG_Consolidations or Nodules 0
QASG_Pleural effusion 0
QAIG_A-line 0
QAIG_B-lines 0
QAIG_Consolidations or Nodules 0
QAIG_Not measured 0
QAIG_Pleural effusion 0
QSLG_A-line 0
QSLG_B-lines 0
QSLG_Consolidations or Nodules 0
QSLG_Not measured 0
QSLG_Pleural effusion 0
QLIG_A-line 0
QLIG_B-lines 0
QLIG_Consolidations or Nodules 0
QLIG_Pleural effusion 0
QPSD_

In [ ]:
l = labels[['record_id', 'TB Label']]
l.to_csv('../CLUSSTER-Benin/labels/justTB.csv')

In [2]:
import pandas as pd




clin_data = pd.read_csv('/home/tjb76/TBLUScopy/Datasets/CLUSSTER-Benin/clinical_data/CLUSSTERBenin-ClinicalDataForResea_DATA_2023-05-24_1630.csv')